# CS-Jail-UR: Exp 2 new models (generation only, GPU)

Runs headless with **Save Version → Save & Run All (Commit)** (limit 12 h; the full run
needs about 3–4 h). Design and the full procedure: `docs/EXP2_NEW_MODELS.md`.

**Rerun of one part only:** set the other flags to `False`. Example: only Gemma's
robustness run → `RUN_R1 = False`, `RUN_GEMMA_MAIN = False`, `RUN_GEMMA_ROBUSTNESS = True`,
`RUN_BACKEND_CHECK = False` (about 1 h).

| Model key | Model | How it is generated |
|---|---|---|
| `r1qwen15` | DeepSeek-R1-Distill-Qwen-1.5B | vLLM fp16, the same stack as qwen25/phi3/llama32; 2,048-token budget |
| `gemma4e2b` | Gemma 4 E2B | Ollama `gemma4:e2b-it-q8_0`, build pinned (vLLM 0.6.3 cannot load Gemma 4) |
| backend check | qwen25 on Ollama | the 200 held-out families, to measure the Ollama-vs-vLLM difference |

Before saving:
1. Settings → Accelerator **GPU T4 x2**, Internet **On**.
2. No secret is required (both models are ungated). Attach `HF_TOKEN` if you have it.
3. Set the flags in the first code cell.

Nothing is judged here. Download `newmodels_outputs.zip` from the version's **Output**
tab and judge on the laptop (signed-in Ollama). Keep this notebook **private**: its
outputs contain harmful prompts and model responses.

In [ ]:
# Set these before "Save & Run All". All vLLM work runs before the Ollama server starts.
RUN_R1 = True                # 3,164 greedy + 2,000 sampled responses (vLLM)
RUN_GEMMA_MAIN = True        # 3,164 greedy responses (Ollama)
RUN_GEMMA_ROBUSTNESS = True  # 2,000 sampled responses (Ollama)
RUN_BACKEND_CHECK = True     # qwen25 via Ollama on the 200 held-out families (800 responses)
RUN_GEMMA = RUN_GEMMA_MAIN or RUN_GEMMA_ROBUSTNESS

import os
from IPython import get_ipython


def sh(cmd):
    """Run a shell command with streamed output; stop the whole run if it fails."""
    print("$", cmd, flush=True)
    get_ipython().system(cmd)
    code = get_ipython().user_ns.get("_exit_code", 0)
    if code:
        raise RuntimeError(f"step failed (exit {code}): {cmd}")


def package():
    """Zip everything produced so far into /kaggle/working/newmodels_outputs.zip (overwrites)."""
    keep = [p for p in ("outputs/exp2/main-r1", "outputs/exp2/robustness-r1", "outputs/exp2/main-gemma4",
                        "outputs/exp2/robustness-gemma4", "outputs/exp2/backend-check-qwen25-ollama")
            if os.path.exists(p)]
    if keep:
        sh("rm -f /kaggle/working/newmodels_outputs.zip && zip -qr /kaggle/working/newmodels_outputs.zip "
           + " ".join(keep))

In [ ]:
# Repo + Python 3.12 environment for the pinned stack (Kaggle's own Python is 3.13).
%cd /kaggle/working
if not os.path.isdir("NLP_Research"):
    sh("git clone --branch revision/final-692-dataset https://github.com/mshamoonbutt/NLP_Research.git")
%cd /kaggle/working/NLP_Research
sh("git pull --ff-only && git log --oneline -1")
sh("python -m pip install -q uv")
sh("python -m uv venv --allow-existing --python 3.12 /root/py312")
# vLLM is only needed for DeepSeek-R1; Ollama-only reruns skip the GPU Python stack.
sh('python -m uv pip install --python /root/py312/bin/python -e "' + ('.[gpu,dev]' if RUN_R1 else '.[local]') + '"')
os.environ["PATH"] = "/root/py312/bin:" + os.environ["PATH"]
os.environ["VIRTUAL_ENV"] = "/root/py312"
os.environ.pop("PYTHONPATH", None)
sh("which python && python --version")

In [ ]:
# GPU, frozen dataset, test suite. Any failure stops the run here.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("No HF_TOKEN attached: fine, both new models are ungated.")
sh("nvidia-smi -L")
sh("python scripts/exp0_finalize_data.py --restore")
sh("python -m pytest tests -q -p no:cacheprovider")

In [ ]:
# DeepSeek-R1-Distill-Qwen-1.5B through vLLM. Its visible reasoning comes before the
# answer, so it gets a 2,048-token budget (the other models: 512).
if RUN_R1:
    sh("python -m csjail.run_eval --models r1qwen15 --max-tokens 2048 --out-dir outputs/exp2/main-r1 --skip-judge")
    package()
    try:  # bounded, so a stall cannot consume the session
        sh("timeout 2h python scripts/exp2_robustness.py --models r1qwen15 --max-tokens 2048 "
           "--greedy-results outputs/exp2/main-r1 --out-dir outputs/exp2/robustness-r1 --skip-judge")
    except RuntimeError as e:
        print("WARN: R1 robustness incomplete (exit 124 = timeout):", e)
    package()

In [ ]:
# Ollama server on the GPU (only after vLLM has exited), then the pinned builds.
# One model in GPU memory at a time: the first run had Gemma and qwen25 loaded together
# and the server started failing (HTTP 500), which emptied Gemma's robustness run.
import subprocess, time, urllib.request


def ollama_up():
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=3)
        return True
    except Exception:
        return False


def ensure_ollama():
    """Start (or restart) the Ollama server if it is not answering."""
    if ollama_up():
        return
    sh("pkill -f 'ollama serve' || true")
    env = dict(os.environ, OLLAMA_NUM_PARALLEL="4", OLLAMA_MAX_LOADED_MODELS="1", OLLAMA_KEEP_ALIVE="10m")
    subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/kaggle/working/ollama.log", "a"),
                     stderr=subprocess.STDOUT)
    for _ in range(90):
        if ollama_up():
            return
        time.sleep(2)
    raise RuntimeError("Ollama server did not start; see /kaggle/working/ollama.log")


if RUN_GEMMA or RUN_BACKEND_CHECK:
    sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd > /dev/null")
    sh("curl -fsSL https://ollama.com/install.sh | sh")
    ensure_ollama()
    os.environ["OLLAMA_WORKERS"] = "4"   # matches OLLAMA_NUM_PARALLEL
    if RUN_GEMMA:
        sh("ollama pull gemma4:e2b-it-q8_0")

In [ ]:
# Gemma 4 E2B through Ollama (production backend for this model; the run refuses a different build).
# Gemma's two runs first, the qwen25 backend check last, so only one model is ever loaded.
if RUN_GEMMA_MAIN:
    ensure_ollama()
    sh("python -m csjail.run_eval --backend ollama --models gemma4e2b --out-dir outputs/exp2/main-gemma4 --skip-judge")
    package()
if RUN_GEMMA_ROBUSTNESS:
    ensure_ollama()
    try:
        sh("timeout 3h python scripts/exp2_robustness.py --backend ollama --models gemma4e2b "
           "--greedy-results outputs/exp2/main-gemma4 --out-dir outputs/exp2/robustness-gemma4 --skip-judge")
    except RuntimeError as e:
        print("WARN: Gemma robustness incomplete (exit 124 = timeout):", e)
    package()
# Backend check: the same qwen25 model through Ollama on the 200 held-out families (a debug run by design).
if RUN_BACKEND_CHECK:
    ensure_ollama()
    sh("ollama pull qwen2.5:1.5b-instruct-q8_0")
    sh("python -m csjail.run_eval --backend ollama --models qwen25 --families eval_main "
       "--out-dir outputs/exp2/backend-check-qwen25-ollama --skip-judge")
    package()

In [ ]:
# Package everything for one download from the Output tab.
package()
sh("ls -lh /kaggle/working/newmodels_outputs.zip")
print("DONE: download newmodels_outputs.zip from this version's Output tab.")